<a href="https://colab.research.google.com/github/xkzy/pdf_scan_merge/blob/main/PDF_Merge_All_Pages.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF Multiply Merge — All Pages → One Page

Upload a PDF, render every page, clean up the scan background, and combine all pages using **Multiply blending** into a single **A4** page.

- Each page's background (tint/shadow/paper texture) is flattened toward white using **brightness-only** correction — hue and saturation are left untouched, so document colors don't shift.
- Every page (including the first) is fit onto an **A4 canvas** preserving its original aspect ratio and **centered** — content is never stretched or resized to fill the page.
- White areas become transparent-like during multiplication.
- Dark/black content accumulates.
- Default rendering resolution: **150 DPI** (change `DPI` below if needed).


In [8]:
!pip -q install pymupdf pillow numpy


In [9]:
from google.colab import files

uploaded = files.upload()
input_pdf = next(iter(uploaded))
print(f"Input: {input_pdf}")



Saving SUPAWIT_ID.pdf to SUPAWIT_ID (2).pdf
Input: SUPAWIT_ID (2).pdf


In [ ]:
import fitz  # PyMuPDF
from PIL import Image, ImageChops, ImageFilter
import numpy as np
import io

# --- Dynamic DPI calculation from first page resolution ---
# Note: input_pdf is expected to be available from a previous cell (e.g., file upload).
_doc_temp = fitz.open(input_pdf)
if len(_doc_temp) == 0:
    _doc_temp.close()
    raise ValueError("The PDF contains no pages.")

_first_page_width_pts = _doc_temp[0].rect.width
_first_page_height_pts = _doc_temp[0].rect.height
_doc_temp.close()

# Target pixel count for the shorter dimension of the rendered first page.
# This provides a good balance for A4 at approximately 300 DPI.
TARGET_SHORTER_DIMENSION_PIXELS = 2400

_shortest_dim_pts = min(_first_page_width_pts, _first_page_height_pts)
if _shortest_dim_pts == 0:
    DPI = 150  # Fallback if page dimensions are zero
else:
    # Calculate DPI such that the shorter dimension of the page renders to TARGET_SHORTER_DIMENSION_PIXELS.
    # DPI = (TARGET_PIXELS * 72 points/inch) / (DIMENSION_IN_POINTS)
    DPI = round((TARGET_SHORTER_DIMENSION_PIXELS * 72) / _shortest_dim_pts)
    # Clamp DPI to a reasonable range for performance and quality balance.
    DPI = max(150, min(DPI, 600))

print(f"Calculated rendering DPI from first page ({_first_page_width_pts}x{_first_page_height_pts} pts): {DPI}")
# --- End dynamic DPI calculation ---

# Remove scan background (tint/shadow/paper texture) before blending.
REMOVE_BACKGROUND = True

# Blur radius (pixels) used to estimate the background for removal.
# Larger = smoother background estimate, better for uneven scan lighting.
BG_BLUR_RADIUS = 25

# Output page orientation: "portrait", "landscape", or "auto"
# ("auto" matches the first PDF page's orientation).
A4_ORIENTATION = "auto"

# Output filename
output_pdf = "multiply_merged.pdf"
A4_WIDTH_MM = 210
A4_HEIGHT_MM = 297

def a4_pixel_size(dpi, orientation, first_page_size):
    w = round(dpi * A4_WIDTH_MM / 25.4)
    h = round(dpi * A4_HEIGHT_MM / 25.4)
    if orientation == "landscape":
        w, h = max(w, h), min(w, h)
    elif orientation == "portrait":
        w, h = min(w, h), max(w, h)
    else:  # auto: match the first page's orientation
        fw, fh = first_page_size
        if fw > fh:
            w, h = max(w, h), min(w, h)
        else:
            w, h = min(w, h), max(w, h)
    return (w, h)

def render_page(page, dpi):
    zoom = dpi / 72.0
    matrix = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=matrix, alpha=False)
    return Image.open(io.BytesIO(pix.tobytes("png"))).convert("RGB")

def remove_scan_background(img, blur_radius=25):
    """Flatten uneven scan background (shadow/paper texture) toward white
    WITHOUT changing document color.

    Only the brightness (HSV 'V' channel) is corrected: a blurred version of
    V is used as a local background estimate and divided out. Hue and
    saturation are left completely untouched, so ink/stamp/highlight colors
    are preserved exactly as scanned.
    """
    hsv = img.convert("HSV")
    h, s, v = hsv.split()

    v_arr = np.asarray(v).astype(np.float32)
    v_bg = np.asarray(v.filter(ImageFilter.GaussianBlur(blur_radius))).astype(np.float32)
    v_bg = np.clip(v_bg, 1, 255)  # avoid division by zero

    v_norm = (v_arr / v_bg) * 255.0
    v_norm = np.clip(v_norm, 0, 255).astype(np.uint8)

    v_new = Image.fromarray(v_norm)
    return Image.merge("HSV", (h, s, v_new)).convert("RGB")

def fit_and_center(img, target_size, fill=(255, 255, 255)):
    """Scale img to fit inside target_size preserving aspect ratio, then
    center it on a white canvas of target_size (no stretching/distortion)."""
    if img.size == target_size:
        return img
    tw, th = target_size
    iw, ih = img.size
    scale = min(tw / iw, th / ih)
    nw, nh = max(1, round(iw * scale)), max(1, round(ih * scale))
    resized = img.resize((nw, nh), Image.Resampling.LANCZOS)
    canvas = Image.new("RGB", target_size, fill)
    offset = ((tw - nw) // 2, (th - nh) // 2)
    canvas.paste(resized, offset)
    return canvas

def prepare_page(page, dpi, remove_bg, blur_radius):
    img = render_page(page, dpi)
    if remove_bg:
        img = remove_scan_background(img, blur_radius)
    return img

doc = fitz.open(input_pdf)
# The 'doc' object is already opened above for DPI calculation.
# No need to check for len(doc) == 0 again.

print(f"Pages: {len(doc)}")

# Render the first page to determine its orientation, then compute the
# fixed A4 canvas size every page will be centered onto.
first_img = prepare_page(doc[0], DPI, REMOVE_BACKGROUND, BG_BLUR_RADIUS)
base_size = a4_pixel_size(DPI, A4_ORIENTATION, first_img.size)
print(f"Output A4 canvas: {base_size[0]} x {base_size[1]} px at {DPI} DPI")

# Fit the first page onto the A4 canvas (no stretching), as the base layer.
result = fit_and_center(first_img, base_size)

for i in range(1, len(doc)):
    print(f"Multiplying page {i+1}/{len(doc)}...")
    layer = prepare_page(doc[i], DPI, REMOVE_BACKGROUND, BG_BLUR_RADIUS)

    # Fit each page onto the same A4 canvas, centered, preserving its
    # original aspect ratio — content size is never disturbed.
    layer = fit_and_center(layer, base_size)

    # PIL ImageChops.multiply implements pixel-wise multiply blending.
    result = ImageChops.multiply(result, layer)

doc.close()

# Save as a single-page A4 PDF.
result.save(output_pdf, "PDF", resolution=DPI)

print(f"\nCreated: {output_pdf}")
print(f"Output size: {result.width} x {result.height} pixels")

Calculated rendering DPI from first page (578.1599731445312x824.4000244140625 pts): 299
Pages: 2
Output A4 canvas: 2472 x 3496 px at 299 DPI
Multiplying page 2/2...

Created: multiply_merged.pdf
Output size: 2472 x 3496 pixels


### New Feature: Document Edge Detection, Cropping, and Alignment

This feature automatically detects the boundaries of the scanned document, crops it, and corrects any perspective distortion before placing it onto the A4 canvas. This ensures that the document content is always perfectly aligned and centered, regardless of how it was scanned.

In [11]:
# Install OpenCV for advanced image processing (edge detection, perspective transform)
!pip -q install opencv-python

In [ ]:
import cv2

# Enable/disable document alignment feature
ENABLE_DOCUMENT_ALIGNMENT = True

# Maximum rotation degrees for alignment (used if automatic rotation is implemented)
MAX_ALIGNMENT_ROTATION_DEGREES = 89

def order_points(pts):
    # initialize a list of coordinates that will be ordered
    # such that the first entry in the list is the top-left,
    # the second entry is the top-right, the third is the
    # bottom-right, and the fourth is the bottom-left
    rect = np.zeros((4, 2), dtype="float32")

    # the top-left point will have the smallest sum, whereas
    # the bottom-right point will have the largest sum
    s = pts.sum(axis=1)
    rect[0] = pts[np.argmin(s)]
    rect[2] = pts[np.argmax(s)]

    # now, compute the difference between the points, the
    # top-right point will have the smallest difference,
    # whereas the bottom-left will have the largest difference
    diff = np.diff(pts, axis=1)
    rect[1] = pts[np.argmin(diff)]
    rect[3] = pts[np.argmax(diff)]

    # return the ordered coordinates
    return rect

def four_point_transform(image, pts):
    # obtain a consistent order of the points and unpack them
    # individually
    rect = order_points(pts)
    (tl, tr, br, bl) = rect

    # compute the width of the new image, which will be the
    # maximum distance between the bottom-right and bottom-left
    # x-coordinates or the top-right and top-left x-coordinates
    widthA = np.sqrt(((br[0] - bl[0]) ** 2) + ((br[1] - bl[1]) ** 2))
    widthB = np.sqrt(((tr[0] - tl[0]) ** 2) + ((tr[1] - tl[1]) ** 2))
    maxWidth = max(int(widthA), int(widthB))

    # compute the height of the new image, which will be the
    # maximum distance between the top-right and bottom-right
    # y-coordinates or the top-left and bottom-left y-coordinates
    heightA = np.sqrt(((tr[0] - br[0]) ** 2) + ((tr[1] - br[1]) ** 2))
    heightB = np.sqrt(((tl[0] - bl[0]) ** 2) + ((tl[1] - bl[1]) ** 2))
    maxHeight = max(int(heightA), int(heightB))

    # now that we have the dimensions of the new image, construct
    # the set of destination points to obtain a "birds eye view",
    # (i.e. top-down view) of the image, again specifying points
    # in the top-left, top-right, bottom-right, and bottom-left
    # order
    dst = np.array([
        [0, 0],
        [maxWidth - 1, 0],
        [maxWidth - 1, maxHeight - 1],
        [0, maxHeight - 1]], dtype="float32")

    # compute the perspective transform matrix and then apply it
    M = cv2.getPerspectiveTransform(rect, dst)
    warped = cv2.warpPerspective(image, M, (maxWidth, maxHeight))

    # return the warped image
    return warped

def find_document_contour_and_corners(image_pil):
    image_cv = np.array(image_pil)
    gray = cv2.cvtColor(image_cv, cv2.COLOR_RGB2GRAY)
    gray = cv2.GaussianBlur(gray, (5, 5), 0)
    edged = cv2.Canny(gray, 75, 200)

    # find the contours in the edged image and initialize the document contour
    cnts = cv2.findContours(edged.copy(), cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
    cnts = cnts[0] if len(cnts) == 2 else cnts[1]
    cnts = sorted(cnts, key=cv2.contourArea, reverse=True)[:5]

    screenCnt = None
    # loop over the contours
    for c in cnts:
        # approximate the contour
        peri = cv2.arcLength(c, True)
        approx = cv2.approxPolyDP(c, 0.02 * peri, True)
        # if our approximated contour has four points, then we can assume that we have found our document
        if len(approx) == 4:
            screenCnt = approx
            break

    if screenCnt is None:
        # No four-point contour found, return original image
        print("Warning: Could not find a 4-point contour for document alignment. Returning original image.")
        return image_pil

    # Apply the four point perspective transform to obtain the top-down view
    warped = four_point_transform(image_cv, screenCnt.reshape(4, 2))
    return Image.fromarray(warped)

# Modify prepare_page to include document alignment
_original_prepare_page = prepare_page # Store original for reference if needed
def prepare_page(page, dpi, remove_bg, blur_radius, enable_alignment=False):
    img = render_page(page, dpi)
    if remove_bg:
        img = remove_scan_background(img, blur_radius)
    if enable_alignment:
        img = find_document_contour_and_corners(img)
    return img

# Update the main loop to use the modified prepare_page with alignment enabled
# Render the first page to determine its orientation, then compute the
# fixed A4 canvas size every page will be centered onto.
first_img = prepare_page(doc[0], DPI, REMOVE_BACKGROUND, BG_BLUR_RADIUS, enable_alignment=ENABLE_DOCUMENT_ALIGNMENT)
base_size = a4_pixel_size(DPI, A4_ORIENTATION, first_img.size)
print(f"Output A4 canvas: {base_size[0]} x {base_size[1]} px at {DPI} DPI")

# Fit the first page onto the A4 canvas (no stretching), as the base layer.
result = fit_and_center(first_img, base_size)

for i in range(1, len(doc)):
    print(f"Multiplying page {i+1}/{len(doc)}...")
    layer = prepare_page(doc[i], DPI, REMOVE_BACKGROUND, BG_BLUR_RADIUS, enable_alignment=ENABLE_DOCUMENT_ALIGNMENT)

    # Fit each page onto the same A4 canvas, centered, preserving its
    # original aspect ratio — content size is never disturbed.
    layer = fit_and_center(layer, base_size)

    # PIL ImageChops.multiply implements pixel-wise multiply blending.
    result = ImageChops.multiply(result, layer)

ValueError: document closed

In [ ]:
from IPython.display import display
display(result)


In [14]:
files.download(output_pdf)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>